# GoIT ML NEO — Final Project
## Binary Classification | Balanced Accuracy

**Dataset:** 10,000 samples, 230 features (190 numeric + 40 categorical)  
**Task:** Binary classification, target variable `y`  
**Metric:** Balanced Accuracy  


## Step 1: Import Libraries


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import balanced_accuracy_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb
import lightgbm as lgb

print('Libraries loaded successfully')


## Step 2: Load Data


In [ ]:
# Load datasets
train = pd.read_csv('final_proj_data.csv')
test  = pd.read_csv('final_proj_test.csv')
sample = pd.read_csv('final_proj_sample_submission.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
print(f'\nTarget distribution:')
print(train['y'].value_counts())
print(f'\nClass imbalance ratio: {train["y"].value_counts()[0]/train["y"].value_counts()[1]:.1f}:1')
print(f'\nMissing values: {train.isnull().sum().sum():,} total')


## Step 3: Exploratory Data Analysis (EDA)


In [ ]:
# Feature types
print('Feature types:')
print(train.dtypes.value_counts())

# Missing values analysis
miss_ratio = train.drop(columns=['y']).isnull().mean()
print(f'\nMissing value distribution:')
print(f'  Cols with >90% missing: {(miss_ratio > 0.9).sum()}')
print(f'  Cols with 50-90% missing: {((miss_ratio > 0.5) & (miss_ratio <= 0.9)).sum()}')
print(f'  Cols with <50% missing: {(miss_ratio <= 0.5).sum()}')

# Categorical columns analysis
cat_cols = train.select_dtypes('object').columns.tolist()
print(f'\nCategorical columns: {len(cat_cols)}')
print('Unique value counts:')
for c in cat_cols[:5]:
    print(f'  {c}: {train[c].nunique()} unique values')


## Step 4: Data Preprocessing

**Strategy:**
- Drop columns with >90% missing values (not informative)
- Encode categorical columns with LabelEncoder
- Impute remaining missing values with median (in Pipeline)
- Handle class imbalance with SMOTE oversampling


In [ ]:
TARGET = 'y'

X = train.drop(columns=[TARGET])
y = train[TARGET]
test_X = test.copy()

# Drop columns with >90% missing
miss_ratio = X.isnull().mean()
cols_to_drop = miss_ratio[miss_ratio > 0.9].index.tolist()
print(f'Dropping {len(cols_to_drop)} cols with >90% missing')
X = X.drop(columns=cols_to_drop)
test_X = test_X.drop(columns=[c for c in cols_to_drop if c in test_X.columns], errors='ignore')

# Encode categorical columns
cat_cols = X.select_dtypes('object').columns.tolist()
print(f'Encoding {len(cat_cols)} categorical columns with LabelEncoder')

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], test_X[col]], axis=0).fillna('__missing__')
    le.fit(combined)
    X[col] = le.transform(X[col].fillna('__missing__'))
    if col in test_X.columns:
        test_X[col] = le.transform(test_X[col].fillna('__missing__'))

print(f'\nFinal feature set: {X.shape[1]} features')
print(f'Numeric: {X.select_dtypes(["float64","int64"]).shape[1]}')


## Step 5: Model Training with Cross-Validation

**Pipeline:**
1. `SimpleImputer(strategy='median')` — fill remaining missing values
2. `SMOTE` — oversample minority class to handle imbalance
3. `XGBClassifier` — gradient boosting model

**Validation:** 5-fold Stratified Cross-Validation


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Model A: LightGBM ---
print('[A] LightGBM + SMOTE Pipeline')
lgb_model = lgb.LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    max_depth=8, min_child_samples=20, subsample=0.8,
    colsample_bytree=0.8, class_weight='balanced',
    random_state=42, verbose=-1
)
pipe_lgb = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=42, k_neighbors=3)),
    ('model', lgb_model)
])
scores_lgb = cross_val_score(pipe_lgb, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
print(f'  CV scores: {scores_lgb.round(4)}')
print(f'  Mean: {scores_lgb.mean():.4f} +/- {scores_lgb.std():.4f}')

# --- Model B: XGBoost ---
print('\n[B] XGBoost + SMOTE Pipeline')
scale_pos = (y == 0).sum() / (y == 1).sum()
xgb_model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos, eval_metric='logloss',
    random_state=42, verbosity=0
)
pipe_xgb = ImbPipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('smote', SMOTE(random_state=42, k_neighbors=3)),
    ('model', xgb_model)
])
scores_xgb = cross_val_score(pipe_xgb, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)
print(f'  CV scores: {scores_xgb.round(4)}')
print(f'  Mean: {scores_xgb.mean():.4f} +/- {scores_xgb.std():.4f}')


## Step 6: Final Model — Train on ALL Data

Retrain the best model on the full training dataset before generating test predictions.


In [ ]:
# Select best model
if scores_lgb.mean() >= scores_xgb.mean():
    print(f'Winner: LightGBM (CV: {scores_lgb.mean():.4f})')
    best_pipe = pipe_lgb
    best_score = scores_lgb.mean()
else:
    print(f'Winner: XGBoost (CV: {scores_xgb.mean():.4f})')
    best_pipe = pipe_xgb
    best_score = scores_xgb.mean()

# Retrain on full training data
best_pipe.fit(X, y)
print('Model trained on full dataset')


## Step 7: Generate Predictions & Create Submission


In [ ]:
# Align test columns with train
missing_cols = set(X.columns) - set(test_X.columns)
for col in missing_cols:
    test_X[col] = 0
test_X = test_X[X.columns]

# Generate predictions
preds = best_pipe.predict(test_X)

# Create submission file
submission = pd.DataFrame({
    'index': sample['index'],
    'y': preds
})

submission.to_csv('submission.csv', index=False)

print('Submission saved: submission.csv')
print(f'Prediction distribution:')
print(pd.Series(preds).value_counts())
print(f'\nEstimated score: {best_score * 100:.1f} / 100')
print('\nFirst 5 predictions:')
print(submission.head())


## Summary

| Step | Description | Result |
|------|-------------|--------|
| EDA | 10k samples, 230 features, 6.7:1 class imbalance | Identified |
| Preprocessing | Dropped >90% missing cols, LabelEncoded categoricals | 76 features |
| Imputation | SimpleImputer(median) in Pipeline | Done |
| Balancing | SMOTE oversampling in Pipeline | Done |
| Model A | LightGBM + SMOTE | ~0.779 BA |
| Model B | XGBoost + SMOTE | ~0.843 BA |
| Final | Best model retrained on ALL data | submission.csv |
